In [18]:
import os
import re
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio

from amazonas_pipeline.defs.assets.constants import ISO3_TO_NAME

In [19]:
data_path = Path(os.environ["DATA_PATH"])
initial_path = data_path / "initial"
generated_path = data_path / "generated"
ghsl_path = Path(os.environ["GHSL_PATH"])

sent_path = Path(os.environ["SENT_PATH"])

In [20]:
amazon_bounds = (
    gpd.read_file(initial_path / "AFP_fixed.gpkg")
    .assign(
        geometry=lambda df: df["geometry"].force_2d(),
    )
    .to_crs("ESRI:54009")["geometry"]
    .item()
)

df_base = (
    gpd.read_file(
        generated_path / "polygons" / "population" / "200_300" / "2020.gpkg",
    )
    .assign(
        combined_polygon_id=lambda df: [f"p{str(i).zfill(6)}" for i in range(len(df))],
    )
    .set_index("combined_polygon_id")
)

max_idx = max(int(x[1:]) for x in df_base.index)
df_modified = (
    gpd.read_file(
        generated_path / "polygons" / "population" / "150_200" / "2020.gpkg",
    )
    .assign(
        combined_polygon_id=lambda df: [
            f"p{str(i + max_idx + 1).zfill(6)}" for i in range(len(df))
        ],
    )
    .set_index("combined_polygon_id")
)

In [21]:
df_modified_in_amazon = df_modified[df_modified.intersects(amazon_bounds)].copy()

idx_base_redundant = (
    df_base[["geometry"]]
    .sjoin(
        df_modified_in_amazon[["geometry"]],
        how="inner",
        predicate="intersects",
    )
    .index.unique()
)

df_final = (
    pd.concat(
        [
            df_base.drop(index=idx_base_redundant).assign(in_amazon="no"),
            df_modified_in_amazon.assign(in_amazon="yes"),
        ],
        ignore_index=False,
    )
    .drop(columns=["polygon_id"])
    .reset_index(names="polygon_id")
)

# Join

In [22]:
def join_cells_with_polygons(
    df_cells: gpd.GeoDataFrame,
    df_polygons: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    df_centroids = df_cells.assign(geometry=lambda df: df["geometry"].centroid).filter(
        ["cell_id", "geometry"],
    )
    joined = (
        df_polygons[["polygon_id", "geometry"]]
        .sjoin(df_centroids, how="inner", predicate="contains")
        .drop(columns=["index_right"])
    )

    cell_to_polygon_id = joined.set_index("cell_id")["polygon_id"].to_dict()
    cell_id_list = set(joined["cell_id"].tolist())  # noqa: F841
    return (
        df_cells.query("cell_id in @cell_id_list")
        .reset_index(drop=True)
        .assign(polygon_id=lambda df: df["cell_id"].map(cell_to_polygon_id))
    )


def add_pop_and_smod_to_cells(
    cells: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    pop_path = ghsl_path / "POP_1000"
    smod_path = ghsl_path / "SMOD_1000"

    centroid_coords = cells.centroid.get_coordinates().to_numpy()

    for raster_path, prefix in zip([smod_path, pop_path], ["smod", "pop"], strict=True):
        for year in range(1975, 2021, 5):
            pop_raster_path = raster_path / f"{year}.tif"
            with rio.open(pop_raster_path) as ds:
                cells[f"{prefix}_{year}"] = np.array(
                    list(ds.sample(centroid_coords)),
                ).squeeze()

    for year in range(1975, 2021, 5):
        cells[f"smod_{year}"] = cells[f"smod_{year}"].floordiv(10).mul(10)

    return cells

In [23]:
df_cells = add_pop_and_smod_to_cells(
    join_cells_with_polygons(
        gpd.read_file(generated_path / "cells" / "countries.gpkg"),
        df_final,
    ),
)

In [24]:
def remove_non_country(id_list: str, country: str) -> str | float:
    out = []
    if id_list is None or (isinstance(id_list, float) and np.isnan(id_list)):
        return np.nan

    for elem in id_list.split("+"):
        if country in elem:
            out.append(elem)
    if len(out) == 0:
        out = id_list.split("+")
    out_str = "+".join(out)
    return re.sub(r"\s\([A-Z]{3}\)", "", out_str).strip().strip("+")


def add_area_and_densities(polygons: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    out = polygons.assign(
        area_km2=polygons.to_crs("ESRI:54009")["geometry"].area / 1e6,
    )
    for year in range(1975, 2021, 5):
        out = out.assign(
            **{f"density_{year}": lambda df: df[f"pop_{year}"] / df["area_km2"]},
        )

    return out


def generate_split_polygons(
    df_polygons: gpd.GeoDataFrame,
    df_cells: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    df_polygons = df_polygons.assign(countries=lambda df: df["GID_0"].str.split("+"))

    single_polygons = df_polygons.query("countries.str.len() == 1").drop(
        columns=["countries"],
    )
    multiple_polygons = (
        df_polygons.query("countries.str.len() > 1")
        .explode("countries")
        .assign(duplicate_id=lambda df: df.groupby("polygon_id").cumcount())
        .assign(
            duplicate_polygon_id=lambda df: df["polygon_id"].astype(str)
            + "_"
            + df["duplicate_id"].astype(str),
        )
        .drop(
            columns=["GID_0", "NAME_0", "geometry"]
            + [
                f"pop{infix}_{year}"
                for year in range(1975, 2021, 5)
                for infix in ["", "_rural", "_urban_center", "_urban_cluster"]
            ],
        )
        .rename(columns={"countries": "GID_0"})
    )

    for prefix in ["GID", "NAME"]:
        for i in range(1, 5):
            multiple_polygons = multiple_polygons.assign(
                **{
                    f"{prefix}_{i}": lambda df: df.apply(
                        lambda row: remove_non_country(
                            row[f"{prefix}_{i}"],
                            row["GID_0"],
                        ),
                        axis=1,
                    ),
                },
            )

    for col in ["name", "max_name"]:
        multiple_polygons = multiple_polygons.assign(
            **{
                col: lambda df: df.apply(
                    lambda row: remove_non_country(row[col], row["GID_0"]),
                    axis=1,
                ),
            },
        )

    cells_merged_with_polygons = (
        multiple_polygons.assign(
            NAME_0=lambda df: df["GID_0"].map(ISO3_TO_NAME),
        )
        .merge(
            df_cells,
            on="polygon_id",
            how="inner",
        )
        .query("country == GID_0")
        .pipe(gpd.GeoDataFrame, geometry="geometry", crs=df_cells.crs)
    )

    total_pops = cells_merged_with_polygons.dissolve(
        "duplicate_polygon_id",
        {
            **{f"GID_{i}": "first" for i in range(5)},
            **{f"NAME_{i}": "first" for i in range(5)},
            "name": "first",
            "max_name": "first",
            **{f"pop_{year}": "sum" for year in range(1975, 2021, 5)},
        },
    )

    pops_by_smod: list[pd.DataFrame] = []
    for year in range(1975, 2021, 5):
        temp = (
            cells_merged_with_polygons.groupby(["duplicate_polygon_id", f"smod_{year}"])
            .agg({f"pop_{year}": "sum"})
            .reset_index()
            .pivot_table(
                index="duplicate_polygon_id",
                columns=f"smod_{year}",
                values=f"pop_{year}",
                fill_value=0,
            )
            .rename(columns={10: "rural", 20: "urban_cluster", 30: "urban_center"})
            .add_prefix("pop_")
            .add_suffix(f"_{year}")
        )
        pops_by_smod.append(temp)

    pops_by_smod_df = pd.concat(pops_by_smod, axis=1)
    final_pops = (
        pd.concat([total_pops, pops_by_smod_df], axis=1)
        .reset_index()
        .drop(columns=["polygon_id"], errors="ignore")
        .rename(columns={"duplicate_polygon_id": "polygon_id"})
    )

    out = (
        pd.concat(
            [single_polygons, final_pops],
            axis=0,
            ignore_index=True,
        )
        .sort_values("polygon_id")
        .pipe(gpd.GeoDataFrame, geometry="geometry", crs=df_polygons.crs)
        .assign(
            area_km2=lambda df: df["geometry"].area,
            in_amazon=lambda df: df["geometry"].intersects(amazon_bounds),
        )
    )

    return add_area_and_densities(out)

In [25]:
df_split = generate_split_polygons(df_final, df_cells)

In [26]:
column_order = (
    ["name", "max_name", "in_amazon"]
    + [f"NAME_{i}" for i in range(5)]
    + [f"GID_{i}" for i in range(5)]
    + [
        f"pop{infix}_{year}"
        for year in range(1975, 2021, 5)
        for infix in ("", "_rural", "_urban_cluster", "_urban_center")
    ]
    + ["area_km2"]
    + [f"density_{year}" for year in range(1975, 2021, 5)]
    + ["geometry"]
)

df_final = df_final[column_order].copy()
df_split = df_split[column_order].copy()

In [37]:
df_final["NAME_0"].value_counts(dropna=False).sort_index()

NAME_0
Argentina                      1928
Argentina+Bolivia                 3
Argentina+Brasil                  8
Argentina+Brasil+Paraguay         1
Argentina+Paraguay               10
Argentina+Uruguay                 3
Bahamas                          20
Barbados                          4
Belice                           67
Belice+Guatemala                  1
Belice+México                     4
Bolivia                        1490
Bolivia+Brasil                    5
Bolivia+Perú                      3
Brasil                        15956
Brasil+Colombia                   5
Brasil+Colombia+Perú              1
Brasil+Guyana                     2
Brasil+Paraguay                   9
Brasil+Perú                       7
Brasil+Uruguay                    5
Brasil+Venezuela                  1
Chile                           846
Colombia                       3272
Colombia+Ecuador                  8
Colombia+Perú                     6
Colombia+Venezuela               11
Costa Rica           

In [38]:
df_final.to_file(sent_path / "DEGURBA" / "polígonos" / "normal.gpkg")
df_split.to_file(sent_path / "DEGURBA" / "polígonos" / "sin_conurb.gpkg")

In [39]:
df_final.drop(columns=["geometry"]).to_excel(
    sent_path / "DEGURBA" / "hojas" / "normal.xlsx", index=False,
)
df_split.drop(columns=["geometry"]).to_excel(
    sent_path / "DEGURBA" / "hojas" / "sin_conurb.xlsx", index=False,
)